<center><h1>Last_First_HW3</h1></center>
<br>
<br>

Name: Li Sumi  <br>
Github Username: SumiLi-1  <br>
USC ID: 5751493481

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data


In [1]:
from pathlib import Path
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.utils import resample

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
RANDOM_STATE = 552
rng = np.random.default_rng(RANDOM_STATE)

Get the AReM Data Set

In [2]:
DATA_DIR = Path('data/AReM')

ACTIVITIES = ['bending1', 'bending2', 'cycling', 'lying', 'sitting', 'standing', 'walking']
SENSOR_COLUMNS = ['avg_rss12', 'var_rss12', 'avg_rss13', 'var_rss13', 'avg_rss23', 'var_rss23']
CSV_COLUMNS = ['time', *SENSOR_COLUMNS]

file_counts = {activity: len(list((DATA_DIR / activity).glob('dataset*.csv')))
               for activity in ACTIVITIES}
pd.Series(file_counts, name='number_of_instances').to_frame()

,number_of_instances
bending1,7
bending2,6
cycling,15
lying,15
sitting,15
standing,15
walking,15


### (b) Test and Train Data

In [3]:
def dataset_number(path: Path) -> int:
    """Return the numeric suffix in a name such as dataset12.csv."""
    match = re.fullmatch(r'dataset(\d+)\.csv', path.name)
    if match is None:
        raise ValueError(f'Unexpected data filename: {path.name}')
    return int(match.group(1))

def is_test_instance(activity: str, number: int) -> bool:
    return number in ({1, 2} if activity.startswith('bending') else {1, 2, 3})

def read_instance(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, skiprows=5, header=None, names=CSV_COLUMNS,
                        sep=r'\s*,\s*|\s+', engine='python', usecols=range(7))
    if frame.shape[1] != 7 or len(frame) not in {479, 480}:
        raise ValueError(f'{path} has unexpected shape {frame.shape}.')
    if len(frame) == 479:
        warnings.warn(f'{path.name} has 479 observations; features use its available values.')
    return frame

instances = []
for activity in ACTIVITIES:
    for path in sorted((DATA_DIR / activity).glob('dataset*.csv'), key=dataset_number):
        number = dataset_number(path)
        instances.append({
            'activity': activity,
            'instance_id': f'{activity}_{number}',
            'dataset_number': number,
            'split': 'test' if is_test_instance(activity, number) else 'train',
            'path': path,
            'series': read_instance(path),
        })

split_summary = (pd.DataFrame([{k: v for k, v in row.items() if k != 'series'} for row in instances])
                 .groupby(['activity', 'split']).size().unstack(fill_value=0))
split_summary.loc['Total'] = split_summary.sum()
split_summary

split,test,train
activity,,
bending1,2,5
bending2,2,4
cycling,3,12
lying,3,12
sitting,3,12
standing,3,12
walking,3,12
Total,19,69


### (c) Feature Extraction

#### i. Research

Common time-domain features include minimum, maximum, range, mean, median, variance, standard deviation, quantiles, interquartile range, root-mean-square, energy, skewness, kurtosis, zero-crossing rate, linear trend/slope, and autocorrelation. This assignment extracts the seven required features: minimum, maximum, mean, median, standard deviation, first quartile, and third quartile.

#### ii. Extraction

In [4]:
FEATURE_FUNCTIONS = {
    'min': lambda x: x.min(),
    'max': lambda x: x.max(),
    'mean': lambda x: x.mean(),
    'median': lambda x: x.median(),
    'std': lambda x: x.std(ddof=1),
    'q1': lambda x: x.quantile(0.25),
    'q3': lambda x: x.quantile(0.75),
}

def extract_features(series: pd.DataFrame) -> dict:
    """Extract 7 required time-domain statistics for each of the 6 sensor series."""
    values = {}
    for sensor in SENSOR_COLUMNS:
        for feature_name, function in FEATURE_FUNCTIONS.items():
            values[f'{sensor}_{feature_name}'] = function(series[sensor])
    return values

feature_rows = []
for instance in instances:
    feature_rows.append({
        'instance_id': instance['instance_id'],
        'activity': instance['activity'],
        'split': instance['split'],
        **extract_features(instance['series']),
    })

features_df = pd.DataFrame(feature_rows)
feature_columns = [column for column in features_df.columns
                   if column not in ['instance_id', 'activity', 'split']]
assert features_df.shape == (88, 45), f'Unexpected feature matrix shape: {features_df.shape}'
assert len(feature_columns) == 42
features_df.head()

,instance_id,activity,split,avg_rss12_min,avg_rss12_max,avg_rss12_mean,avg_rss12_median,avg_rss12_std,avg_rss12_q1,avg_rss12_q3,...,avg_rss23_std,avg_rss23_q1,avg_rss23_q3,var_rss23_min,var_rss23_max,var_rss23_mean,var_rss23_median,var_rss23_std,var_rss23_q1,var_rss23_q3
0,bending1_1,bending1,test,37.25,45.00,40.624792,40.50,1.476967,39.25,42.00,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30
1,bending1_2,bending1,test,38.00,45.67,42.812812,42.50,1.435550,42.00,43.67,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30
2,bending1_3,bending1,train,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94
3,bending1_4,bending1,train,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00
4,bending1_5,bending1,train,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50


#### iii. Standard Deviation

In [5]:
def bootstrap_std_ci(values, confidence=0.90, n_resamples=10_000, random_state=RANDOM_STATE):
    """Percentile bootstrap confidence interval for a sample standard deviation."""
    values = np.asarray(values, dtype=float)
    local_rng = np.random.default_rng(random_state)
    sample_indices = local_rng.integers(0, values.size, size=(n_resamples, values.size))
    bootstrap_statistics = values[sample_indices].std(axis=1, ddof=1)
    alpha = 1 - confidence
    return values.std(ddof=1), *np.quantile(bootstrap_statistics, [alpha / 2, 1 - alpha / 2])

ci_rows = []
for feature in feature_columns:
    estimate, lower, upper = bootstrap_std_ci(features_df[feature])
    ci_rows.append({
        'feature': feature,
        'standard_deviation': estimate,
        'ci_90_lower': lower,
        'ci_90_upper': upper,
    })

bootstrap_summary = pd.DataFrame(ci_rows).sort_values('feature').reset_index(drop=True)
bootstrap_summary

,feature,standard_deviation,ci_90_lower,ci_90_upper
0,avg_rss12_max,4.394362,3.328017,5.284915
1,avg_rss12_mean,5.335718,4.704031,5.886357
2,avg_rss12_median,5.440054,4.790278,6.004313
3,avg_rss12_min,9.569975,8.262167,10.761378
4,avg_rss12_q1,6.153590,5.581620,6.652410
5,avg_rss12_q3,5.138925,4.331936,5.840872
6,avg_rss12_std,1.772153,1.572160,1.950031
7,avg_rss13_max,4.875137,4.170186,5.457330
8,avg_rss13_mean,4.008380,3.416303,4.491822
9,avg_rss13_median,4.036396,3.438837,4.535112


#### iv. Select Features

I select **mean**, **standard deviation**, and **maximum**. Mean captures the typical signal strength, standard deviation captures motion-related variability, and maximum preserves a useful extreme value. The code below summarizes their separation across the seven activity classes.

In [6]:
selected_statistics = ['mean', 'std', 'max']
selected_features = [f'{sensor}_{statistic}'
                     for sensor in SENSOR_COLUMNS for statistic in selected_statistics]
features_df.groupby('activity')[selected_features].agg(['mean', 'std']).round(3)

avg_rss12_mean        avg_rss12_std        avg_rss12_max         \
                   mean    std          mean    std          mean    std   
activity                                                                   
bending1         42.668  1.250         1.913  0.827        46.796  1.276   
bending2         29.415  2.920         4.538  2.275        44.958  6.581   
cycling          36.747  0.521         3.773  0.393        44.483  0.476   
lying            39.850  7.142         2.025  2.509        42.733  8.640   
sitting          41.749  3.424         2.052  1.505        46.356  2.909   
standing         43.858  1.415         2.189  0.512        47.120  0.916   
walking          34.434  0.458         4.789  0.203        46.261  2.275   

         var_rss12_mean        var_rss12_std         ... avg_rss23_std         \
                   mean    std          mean    std  ...          mean    std   
activity                                             ...                        
bending1          0.454  0.122         0.366  0.125  ...         2.607  0.709   
bending2          0.448  0.113         0.662  0.211  ...         4.871  2.052   
cycling           2.688  0.364         1.893  0.214  ...         3.025  0.317   
lying             0.343  0.185         0.287  0.172  ...         3.598  0.663   
sitting           0.401  0.136         0.371  0.155  ...         4.437  1.102   
standing          0.399  0.072         0.368  0.051  ...         3.430  0.689   
walking           4.337  0.148         2.439  0.114  ...         3.099  0.121   

         avg_rss23_max        var_rss23_mean        var_rss23_std         \
                  mean    std           mean    std          mean    std   
activity                                                                   
bending1        36.524  3.131          0.552  0.094         0.532  0.080   
bending2        32.722  5.777          0.982  0.219         0.905  0.302   
cycling         24.489  1.292          3.037  0.229         1.749  0.111   
lying           17.595  3.666          0.661  0.072         0.560  0.119   
sitting         25.200  2.429          0.903  0.155         0.821  0.150   
standing        21.078  1.208          0.913  0.157         0.710  0.143   
walking         24.183  0.956          3.344  0.087         1.700  0.057   

         var_rss23_max         
                  mean    std  
activity                       
bending1         2.716  1.246  
bending2         7.232  3.234  
cycling          9.175  0.747  
lying            4.041  0.983  
sitting          6.094  1.146  
standing         5.313  1.075  
walking          9.127  0.654  

[7 rows x 36 columns]

## 2. ISLR 3.7.4

We compare a linear regression with a cubic regression. The cubic model contains the linear model as the special case $\beta_2=\beta_3=0$.

### (a) Linear Train

Because cubic regression is more flexible and nests the linear model, its training RSS must be less than or equal to the linear model's training RSS. It can use the extra $X^2$ and $X^3$ terms to fit residual noise, whereas the linear model cannot.

### (b) Linear Test

When the true relationship is linear, the linear regression is expected to have lower test RSS. The cubic model adds unnecessary flexibility: it has no bias advantage here but has higher variance, so it is more prone to fitting training noise rather than the underlying linear relationship.

### (c) Not Linear Train

The cubic regression again has training RSS less than or equal to the linear regression's training RSS. This conclusion does not require the true relationship to be linear: on a fixed training set, the cubic model has every linear-model option plus additional terms.

### (d) Not Linear Testing

There is not enough information to determine which model has lower test RSS. A cubic model may reduce bias if the true relationship is sufficiently non-linear, but it also increases variance. The outcome depends on how non-linear the true relationship is, the noise level, and the sample size.